# © Artur Czarnecki. All rights reserved.

## Echo experiment — end-to-end §35 smoke test

Validates the full laboratory loop with **EchoAgent** (deterministic, no network):

register → NexusLoop → trace → evaluate → decide

In [ ]:
import asyncio
from pathlib import Path

from intergrax.experiments.models import ExperimentDecision, RegisterExperimentRequest
from intergrax.experiments.workflow import ExperimentSession, ensure_repo_root_on_path
from intergrax.runtime.registry.bootstrap import build_harness_registry
from intergrax.runtime.task.task import TaskState

REPO_ROOT = ensure_repo_root_on_path()
BUILD_DIR = REPO_ROOT / "build" / "notebooks"
BUILD_DIR.mkdir(parents=True, exist_ok=True)

session = ExperimentSession(
    experiments_db=BUILD_DIR / "echo_experiments.db",
    trace_db=BUILD_DIR / "echo_trace.db",
    tenant_id="t1",
    user_id="u1",
)

record = session.register(
    RegisterExperimentRequest(
        hypothesis="EchoAgent returns deterministic prefixed answer via NexusLoop",
        capability="echo.basic",
        agent_id="echo",
        expected_output="hello echo lab",
        validation_criteria="non-empty answer containing user message",
    )
)
print(f"experiment_id: {record.experiment_id}")

In [ ]:
loop = session.build_nexus_loop(build_harness_registry(include_echo=True))


async def _run():
    return await session.run(
        loop=loop,
        record=record,
        message="hello echo lab",
    )


outcome = asyncio.run(_run())

assert outcome.task_result.state == TaskState.COMPLETED
assert "hello echo lab" in outcome.task_result.answer
assert outcome.passed

print("answer:", outcome.task_result.answer)
print("run_id:", outcome.task_result.run_id)
print("checks:", outcome.checks)
print("trace events:", outcome.trace_event_count)

In [ ]:
run_id = outcome.task_result.run_id or outcome.task_result.task_id
summary = session.summarize_trace(run_id)
print("trace summary:", summary)

final = session.decide(
    record.experiment_id,
    ExperimentDecision.KEEP,
    notes="Echo smoke test passed in notebook",
)
print(f"decision: {final.decision.value}")
print(f"linked runs: {final.run_ids}")
print("ECHO EXPERIMENT OK")